In [ ]:
import sympy as sp 

In [ ]:
# c21 vec
c21str = ''
for i in range(6):
    for j in range(i,6):
        c21str += f"c{i+1}{j+1}" + " "
c21vec = sp.symbols(c21str)
c66 = sp.Array.zeros(6,6)
c66 = c66.as_mutable()
for i in range(6):
    for j in range(i,6):
        c66[i,j] = c21vec[i*6+j-i*(i+1)//2]
        c66[j,i] = c66[i,j]

cbak = c66 * 1
eps = sp.Rational(1,3) * (c66[0,0] + c66[1,1] + c66[2,2])
mu = sp.Rational(1,3) * (c66[3,3] + c66[4,4] + c66[5,5])
kappa = eps - sp.Rational(4,3) * mu 

Qk_fac,Qmu_fac = sp.symbols("Qk_fac,Qmu_fac")

for i in range(3):
    cbak[i,i] = c66[i,i] - eps + kappa * Qk_fac + sp.Rational(4,3) * mu * Qmu_fac

for i in range(3):
    for j in range(3):
        if i == j: 
            continue
        cbak[i,j] = c66[i,j] - eps + kappa * Qk_fac + 2 * mu * (1 - sp.Rational(1,3) * Qmu_fac)

for i in range(3,6):
    cbak[i,i] = c66[i,i] * Qmu_fac

In [ ]:
id = 0
for i in range(6):
    for j in range(i,6):
        id1 = 0
        for i1 in range(6): 
            for j1 in range(i1,6):
                expr = sp.diff(cbak[i,j],c66[i1,j1])
                if expr != 0:
                    code = sp.ccode(expr)
                    print(f"SETDC({id},{id1},{code});")
                id1 += 1
        id += 1

In [ ]:
id = 0
for i in range(6):
    for j in range(i,6):
        for i1,p in enumerate([Qk_fac,Qmu_fac]):
            expr = sp.diff(cbak[i,j],p)
            if expr != 0:
                code = sp.ccode(expr)
                if i1 == 0:
                    code = f"({code}) * dQk_fac"
                else:
                    code = f"({code}) * dQmu_fac"
                
                for jj in range(1,7):
                    for kk in range(jj,7):
                        code = code.replace(f"c{jj}{kk}",f"C({jj-1},{kk-1})*1.")
                print(f"SETDQ({id},{i1},{code});")
        id += 1